# Phase 4 — Meta-Learner Training & Evaluation

Trains two meta-learners (kNN, MLP) on each meta-feature representation and evaluates
with Leave-One-Out cross-validation (LOO-CV, n=51).

**Primary metric**: Top-1 accuracy — did the predicted best method actually rank first?  
**Secondary metric**: LSE-MAE — when predicting all 6 LSE values, mean absolute error  
**Baselines**:
- Lower bound: always predict k-means (most common, 14/51 = 27%)
- Upper bound: oracle (always correct, 100%)

**Ablation**: compares Option A (hand-crafted) vs B (autoencoder) vs C (dict learning)  
**Output**: saved models in `outputs/models/`

In [1]:
import os, sys, warnings, pickle
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import LeaveOneOut

def find_project_root(start=None):
    cur = Path(start or os.getcwd()).resolve()
    for path in (cur, *cur.parents):
        if (path / 'CLAUDE.md').exists() and (path / 'src').exists():
            return str(path)
    raise RuntimeError('Project root not found; run from the repo root or notebooks folder')

ROOT = find_project_root()
if os.path.join(ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))

from meta_learner import (
    extract_Xy_clf, extract_Xy_reg,
    loo_classify, loo_regress,
    baseline_always, oracle_expected_lse,
    report_clf_results, LSE_COLS, METHOD_NAMES,
)

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [2]:
# ── Load the three feature tables ─────────────────────────────────────────────
def prefer_fixed_csv(fixed_name, legacy_name):
    fixed_path = os.path.join(META_DIR, fixed_name)
    legacy_path = os.path.join(META_DIR, legacy_name)
    return fixed_path if os.path.exists(fixed_path) else legacy_path

df_a = pd.read_csv(prefer_fixed_csv('meta_training_optA_fixed.csv', 'meta_training_optA.csv'))
df_b = pd.read_csv(prefer_fixed_csv('meta_training_optB_fixed.csv', 'meta_training_optB.csv'))
df_c = pd.read_csv(prefer_fixed_csv('meta_training_optC_fixed.csv', 'meta_training_optC.csv'))

# Drop rows where best_method is NaN (fully failed datasets)
df_a = df_a.dropna(subset=['best_method']).reset_index(drop=True)
df_b = df_b.dropna(subset=['best_method']).reset_index(drop=True)
df_c = df_c.dropna(subset=['best_method']).reset_index(drop=True)

print(f'Option A: {df_a.shape}  Option B: {df_b.shape}  Option C: {df_c.shape}')
print(f'Best method distribution (Option A):')
print(df_a['best_method'].value_counts().to_string())

Option A: (42, 27)  Option B: (42, 51)  Option C: (42, 51)
Best method distribution (Option A):
best_method
dbscan       16
kmeans        8
gmm           7
agg           5
autoenc       4
dictlearn     2


In [3]:
# ── Baselines ─────────────────────────────────────────────────────────────────
y_true = df_a['best_method'].values

baseline_kmeans = baseline_always('kmeans', y_true)
baseline_dbscan = baseline_always('dbscan', y_true)  # most frequent in our data
oracle_acc      = 1.0
oracle_lse      = oracle_expected_lse(df_a)

# Expected LSE if always picking k-means (regardless of whether it's best)
always_kmeans_lse = df_a['LSE_kmeans'].mean()

print('=== Baselines ===')
print(f'  Always k-means accuracy   : {baseline_kmeans:.3f}  (expected LSE: {always_kmeans_lse:.3f})')
print(f'  Always dbscan  accuracy   : {baseline_dbscan:.3f}')
print(f'  Oracle accuracy           : {oracle_acc:.3f}  (expected LSE: {oracle_lse:.3f})')

=== Baselines ===
  Always k-means accuracy   : 0.190  (expected LSE: 0.628)
  Always dbscan  accuracy   : 0.381
  Oracle accuracy           : 1.000  (expected LSE: 0.739)


## kNN — k Sweep on Option A

In [4]:
# ── Find best k via LOO-CV on Option A ────────────────────────────────────────
X_a, y_a, feat_cols_a, ids_a = extract_Xy_clf(df_a)

k_candidates = [1, 3, 5, 7, 9, 11]
k_results = {}

for k in k_candidates:
    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale',  StandardScaler()),
        ('clf',    KNeighborsClassifier(n_neighbors=k)),
    ])
    res = loo_classify(pipe, X_a, y_a)
    k_results[k] = res['accuracy']
    print(f'  k={k:2d}  accuracy={res["accuracy"]:.3f}')

best_k = max(k_results, key=k_results.get)
print(f'\nBest k = {best_k}  (accuracy={k_results[best_k]:.3f})')

  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.

  k= 1  accuracy=0.190
  k= 3  accuracy=0.262
  k= 5  accuracy=0.214
  k= 7  accuracy=0.238
  k= 9  accuracy=0.286
  k=11  accuracy=0.405

Best k = 11  (accuracy=0.405)


## Classification LOO-CV — All Models × All Options

In [5]:
# ── Define final pipelines ────────────────────────────────────────────────────
knn_clf = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('clf',    KNeighborsClassifier(n_neighbors=best_k)),
])

mlp_clf = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('clf',    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        random_state=SEED,
    )),
])

print('Pipelines defined')

Pipelines defined


In [6]:
# ── Run classification LOO-CV for all 6 combos ────────────────────────────────
options = {
    'A_handcrafted': df_a,
    'B_autoencoder': df_b,
    'C_dictlearn'  : df_c,
}

clf_results = {}  # (option, model) -> result dict

for opt_name, df_opt in options.items():
    X_opt, y_opt, _, _ = extract_Xy_clf(df_opt)
    for model_name, pipe in [('kNN', knn_clf), ('MLP', mlp_clf)]:
        key = f'{opt_name} / {model_name}'
        print(f'Running {key} ...')
        res = loo_classify(pipe, X_opt, y_opt)
        clf_results[key] = res
        report_clf_results(key, res)

print('\nDone.')

Running A_handcrafted / kNN ...
  A_handcrafted / kNN                       Top-1 acc = 0.405  pred_dist={'dbscan': 37, 'agg': 2, 'kmeans': 3}
Running A_handcrafted / MLP ...
  A_handcrafted / MLP                       Top-1 acc = 0.286  pred_dist={'dbscan': 19, 'gmm': 7, 'autoenc': 3, 'dictlearn': 3, 'kmeans': 6, 'agg': 4}
Running B_autoencoder / kNN ...
  B_autoencoder / kNN                       Top-1 acc = 0.333  pred_dist={'dbscan': 36, 'kmeans': 4, 'gmm': 1, 'agg': 1}
Running B_autoencoder / MLP ...
  B_autoencoder / MLP                       Top-1 acc = 0.333  pred_dist={'dbscan': 16, 'gmm': 7, 'kmeans': 6, 'autoenc': 3, 'dictlearn': 5, 'agg': 5}
Running C_dictlearn / kNN ...
  C_dictlearn / kNN                         Top-1 acc = 0.310  pred_dist={'agg': 4, 'dbscan': 37, 'kmeans': 1}
Running C_dictlearn / MLP ...
  C_dictlearn / MLP                         Top-1 acc = 0.286  pred_dist={'autoenc': 4, 'dbscan': 19, 'kmeans': 7, 'agg': 3, 'gmm': 7, 'dictlearn': 2}

Done.


In [7]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for key, res in clf_results.items():
    opt, model = key.split(' / ')
    rows.append({'Option': opt, 'Model': model, 'Top-1 Accuracy': round(res['accuracy'], 3)})

# Add baselines
rows.append({'Option': '—', 'Model': 'Always k-means',  'Top-1 Accuracy': round(baseline_kmeans, 3)})
rows.append({'Option': '—', 'Model': 'Always dbscan',   'Top-1 Accuracy': round(baseline_dbscan, 3)})
rows.append({'Option': '—', 'Model': 'Oracle (upper)',  'Top-1 Accuracy': 1.000})

summary_df = pd.DataFrame(rows)
print('=== Classification LOO-CV Results ===')
print(summary_df.sort_values('Top-1 Accuracy', ascending=False).to_string(index=False))

=== Classification LOO-CV Results ===
       Option          Model  Top-1 Accuracy
            — Oracle (upper)           1.000
A_handcrafted            kNN           0.405
            —  Always dbscan           0.381
B_autoencoder            kNN           0.333
B_autoencoder            MLP           0.333
  C_dictlearn            kNN           0.310
A_handcrafted            MLP           0.286
  C_dictlearn            MLP           0.286
            — Always k-means           0.190


## Detailed Analysis — Best Classifier (Option A)

In [8]:
# Pick the best model on Option A to inspect in detail
knn_a_res = clf_results['A_handcrafted / kNN']
mlp_a_res = clf_results['A_handcrafted / MLP']
best_a_res = knn_a_res if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else mlp_a_res
best_a_name = 'kNN' if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else 'MLP'

print(f'=== Best on Option A: {best_a_name} (acc={best_a_res["accuracy"]:.3f}) ===')
print('\nClassification report:')
print(classification_report(best_a_res['trues'], best_a_res['preds'], zero_division=0))

print('\nConfusion matrix (rows=true, cols=pred):')
labels = sorted(df_a['best_method'].unique())
cm = confusion_matrix(best_a_res['trues'], best_a_res['preds'], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())

=== Best on Option A: kNN (acc=0.405) ===

Classification report:
              precision    recall  f1-score   support

         agg       0.00      0.00      0.00         5
     autoenc       0.00      0.00      0.00         4
      dbscan       0.43      1.00      0.60        16
   dictlearn       0.00      0.00      0.00         2
         gmm       0.00      0.00      0.00         7
      kmeans       0.33      0.12      0.18         8

    accuracy                           0.40        42
   macro avg       0.13      0.19      0.13        42
weighted avg       0.23      0.40      0.26        42


Confusion matrix (rows=true, cols=pred):
           agg  autoenc  dbscan  dictlearn  gmm  kmeans
agg          0        0       4          0    0       1
autoenc      1        0       3          0    0       0
dbscan       0        0      16          0    0       0
dictlearn    0        0       2          0    0       0
gmm          0        0       6          0    0       1
kmeans       

In [9]:
# ── Which datasets were mis-predicted? ────────────────────────────────────────
misses = [
    {
        'dataset_id': ids_a[i],
        'true':       best_a_res['trues'][i],
        'predicted':  best_a_res['preds'][i],
        'LSE_true_method':  df_a.loc[i, f'LSE_{best_a_res["trues"][i]}'],
        'LSE_pred_method':  df_a.loc[i, f'LSE_{best_a_res["preds"][i]}'],
    }
    for i in range(len(ids_a))
    if not best_a_res['correct_mask'][i]
]

miss_df = pd.DataFrame(misses)
if len(miss_df) > 0:
    miss_df['LSE_cost'] = (miss_df['LSE_true_method'] - miss_df['LSE_pred_method']).round(3)
    print(f'=== {len(miss_df)} mis-predictions ===')
    print(miss_df.to_string(index=False))
    print(f'\nMean LSE cost of errors: {miss_df["LSE_cost"].mean():.3f}')
    print(f'(= how much LSE is lost by picking the wrong method)')
else:
    print('Perfect prediction — no errors!')

=== 25 mis-predictions ===
 dataset_id      true predicted  LSE_true_method  LSE_pred_method  LSE_cost
      43924    kmeans    dbscan         0.630435         0.630435     0.000
        732    kmeans    dbscan         0.707317         0.707317     0.000
       1459       gmm    dbscan         0.260661         0.245736     0.015
       1495    kmeans    dbscan         1.000000         1.000000     0.000
       4538   autoenc       agg         0.536419         0.494689     0.042
      46176    kmeans    dbscan         0.987097         0.451613     0.535
      43958       agg    kmeans         0.683351         0.568009     0.115
      42544       gmm    dbscan         0.541667         0.187500     0.354
      45035    kmeans       agg         0.790828         0.758867     0.032
         39       agg    dbscan         1.000000         0.813559     0.186
      44033    kmeans    dbscan         0.645491         0.593057     0.052
      44367    kmeans    dbscan         0.672897         0.62

## Regression Variant — Predict All 6 LSE Values

In [10]:
knn_reg = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('reg',    KNeighborsRegressor(n_neighbors=best_k)),
])

mlp_reg = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale',  StandardScaler()),
    ('reg',    MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        random_state=SEED,
    )),
])

print('Regression pipelines defined')

Regression pipelines defined


In [11]:
# ── Regression LOO-CV on Option A only (main approach) ────────────────────────
X_a_reg, Y_a_reg, _, _ = extract_Xy_reg(df_a)

reg_results = {}
for model_name, pipe in [('kNN', knn_reg), ('MLP', mlp_reg)]:
    print(f'Running regression {model_name} ...')
    res = loo_regress(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results[model_name] = res
    print(f'  MAE mean: {res["mae_mean"]:.4f}')
    print(f'  MAE per method: ' + '  '.join(
        f'{m}={v:.3f}' for m, v in zip(METHOD_NAMES, res['mae_per_col'])
    ))
    print(f'  Argmax accuracy (predict best via argmax of predicted LSE): {res["argmax_accuracy"]:.3f}')
    print(f'  Mean expected LSE of predicted method: {res["expected_lse"]:.3f}')
    print()

print(f'Oracle expected LSE (upper bound): {oracle_lse:.3f}')
print(f'Always k-means expected LSE:       {always_kmeans_lse:.3f}')

Running regression kNN ...
  MAE mean: 0.1447
  MAE per method: kmeans=0.156  dbscan=0.149  agg=0.147  gmm=0.151  autoenc=0.141  dictlearn=0.124
  Argmax accuracy (predict best via argmax of predicted LSE): 0.238
  Mean expected LSE of predicted method: 0.614

Running regression MLP ...
  MAE mean: 118.3604
  MAE per method: kmeans=189.443  dbscan=43.547  agg=63.936  gmm=232.973  autoenc=122.553  dictlearn=57.711
  Argmax accuracy (predict best via argmax of predicted LSE): 0.167
  Mean expected LSE of predicted method: 0.627

Oracle expected LSE (upper bound): 0.739
Always k-means expected LSE:       0.628


## Ablation Summary — Option A vs B vs C

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

abl_rows = []
for opt_name in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']:
    for model_name in ['kNN', 'MLP']:
        key  = f'{opt_name} / {model_name}'
        acc  = clf_results[key]['accuracy']
        abl_rows.append({'Option': opt_name, 'Model': model_name, 'Accuracy': acc})

abl_df = pd.DataFrame(abl_rows)
print('=== Ablation: Top-1 Classification Accuracy ===')
pivot = abl_df.pivot(index='Option', columns='Model', values='Accuracy').round(3)
print(pivot.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.35
knn_accs = [abl_df.loc[(abl_df.Option == o) & (abl_df.Model == 'kNN'), 'Accuracy'].values[0]
            for o in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']]
mlp_accs = [abl_df.loc[(abl_df.Option == o) & (abl_df.Model == 'MLP'), 'Accuracy'].values[0]
            for o in ['A_handcrafted', 'B_autoencoder', 'C_dictlearn']]

ax.bar(x - w/2, knn_accs, w, label='kNN', color='steelblue')
ax.bar(x + w/2, mlp_accs, w, label='MLP', color='coral')
ax.axhline(baseline_kmeans, ls='--', color='gray', label=f'Always k-means ({baseline_kmeans:.2f})')
ax.axhline(oracle_acc,      ls=':',  color='black', label='Oracle (1.0)')
ax.set_xticks(x)
ax.set_xticklabels(['A: Hand-crafted', 'B: Autoencoder', 'C: Dict Learning'])
ax.set_ylabel('Top-1 Accuracy (LOO-CV)')
ax.set_title('Meta-learner ablation — feature representation comparison')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
fig_path = os.path.join(FIGS_DIR, 'ablation_accuracy.png')
fig.savefig(fig_path, dpi=120)
plt.close()
print(f'Figure saved → {fig_path}')

=== Ablation: Top-1 Classification Accuracy ===
Model            MLP    kNN
Option                     
A_handcrafted  0.286  0.405
B_autoencoder  0.333  0.333
C_dictlearn    0.286  0.310
Figure saved → C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\figures\ablation_accuracy.png


## Save Best Models

In [13]:
# ── Train final models on full dataset (no held-out fold) ─────────────────────

# ---- Classifier ----
best_clf_pipe = knn_clf if knn_a_res['accuracy'] >= mlp_a_res['accuracy'] else mlp_clf

final_clf = clone(best_clf_pipe)
final_clf.fit(X_a, y_a)

clf_path = os.path.join(MODELS_DIR, 'meta_clf_optA.pkl')
with open(clf_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf, 'feature_cols': feat_cols_a}, f)
print(f'Classifier saved → {clf_path}  ({best_a_name})')

# ---- Regressor ----
best_reg_name = 'kNN' if reg_results['kNN']['mae_mean'] <= reg_results['MLP']['mae_mean'] else 'MLP'
best_reg_pipe = knn_reg if best_reg_name == 'kNN' else mlp_reg

final_reg = clone(best_reg_pipe)
valid_mask = ~np.isnan(Y_a_reg).any(axis=1)
final_reg.fit(X_a[valid_mask], Y_a_reg[valid_mask])

reg_path = os.path.join(MODELS_DIR, 'meta_reg_optA.pkl')
with open(reg_path, 'wb') as f:
    pickle.dump({
        'pipeline': final_reg,
        'feature_cols': feat_cols_a,
        'lse_cols': LSE_COLS,
        'method_names': METHOD_NAMES,
    }, f)
print(f'Regressor  saved → {reg_path}  ({best_reg_name})')

# ---- Save best B and C classifiers for ablation/app use ----
for letter, full_key_prefix, df_opt in [
    ('B', 'B_autoencoder', df_b),
    ('C', 'C_dictlearn',   df_c),
]:
    X_opt, y_opt, feat_opt, _ = extract_Xy_clf(df_opt)
    knn_acc_opt = clf_results[f'{full_key_prefix} / kNN']['accuracy']
    mlp_acc_opt = clf_results[f'{full_key_prefix} / MLP']['accuracy']
    best_pipe_opt = knn_clf if knn_acc_opt >= mlp_acc_opt else mlp_clf
    final_clf_opt = clone(best_pipe_opt)
    final_clf_opt.fit(X_opt, y_opt)
    path_opt = os.path.join(MODELS_DIR, f'meta_clf_opt{letter}.pkl')
    with open(path_opt, 'wb') as f:
        pickle.dump({'pipeline': final_clf_opt, 'feature_cols': feat_opt}, f)
    print(f'Saved opt{letter} clf → {path_opt}')

Classifier saved → C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optA.pkl  (kNN)
Regressor  saved → C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\models\meta_reg_optA.pkl  (kNN)
Saved optB clf → C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optB.pkl
Saved optC clf → C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optC.pkl


## Final Summary

In [14]:
print('=' * 60)
print('PHASE 4 SUMMARY')
print('=' * 60)
print(f'  n_datasets        : {len(df_a)}')
print(f'  Best k (kNN)      : {best_k}')
print()
print('  CLASSIFICATION (LOO-CV Top-1 Accuracy)')
print(f'  Lower bound       : {baseline_kmeans:.3f}  (always k-means)')
print(f'  Upper bound       : {oracle_acc:.3f}  (oracle)')
for key in [
    'A_handcrafted / kNN', 'A_handcrafted / MLP',
    'B_autoencoder / kNN', 'B_autoencoder / MLP',
    'C_dictlearn / kNN',   'C_dictlearn / MLP',
]:
    acc = clf_results[key]['accuracy']
    print(f'  {key:35s}: {acc:.3f}')
print()
print('  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)')
for model_name, res in reg_results.items():
    print(f'  {model_name:5s}  MAE={res["mae_mean"]:.4f}  '
          f'argmax_acc={res["argmax_accuracy"]:.3f}  '
          f'expected_LSE={res["expected_lse"]:.3f}')
print()
print(f'  Models saved to: {MODELS_DIR}')
print()
print('Phase 4 complete. Ready for Phase 5 (SHAP analysis).')

PHASE 4 SUMMARY
  n_datasets        : 42
  Best k (kNN)      : 11

  CLASSIFICATION (LOO-CV Top-1 Accuracy)
  Lower bound       : 0.190  (always k-means)
  Upper bound       : 1.000  (oracle)
  A_handcrafted / kNN                : 0.405
  A_handcrafted / MLP                : 0.286
  B_autoencoder / kNN                : 0.333
  B_autoencoder / MLP                : 0.333
  C_dictlearn / kNN                  : 0.310
  C_dictlearn / MLP                  : 0.286

  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)
  kNN    MAE=0.1447  argmax_acc=0.238  expected_LSE=0.614
  MLP    MAE=118.3604  argmax_acc=0.167  expected_LSE=0.627

  Models saved to: C:\Users\adamj\Desktop\LAU\Masters\Spring 2026\Machine Learning\Project\ml-pseudo-label-meta-learning\outputs\models

Phase 4 complete. Ready for Phase 5 (SHAP analysis).
